# ML-05 — Feature Vector and Leakage/Privacy Check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Melih-Yilmaz06/flyrank-ml-internship/blob/main/work/notebooks/w03_feature_leakage_check.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Build the feature vector

*Code that actually builds it — engineered features, categorical handling, fills.*

---

**Lane:** Refresh / Content Opportunity Scoring

We query the `fact_content_daily_performance` table for `month='2026-03'`, filter to `gsc_data_available IS TRUE`, aggregate to one row per URL, and engineer five numeric features. We also join `dim_content` to pull `title` length as a static metadata feature. Missing numeric values are filled with 0; no categorical columns enter the model.

In [ ]:
!pip install -q duckdb pandas scikit-learn

import os, duckdb, pandas as pd, numpy as np
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import cross_val_score, GroupKFold

try:
    from google.colab import userdata
    HF_TOKEN = userdata.get('HF_TOKEN')
except Exception:
    HF_TOKEN = os.environ.get('HF_TOKEN')
assert HF_TOKEN, 'Set HF_TOKEN as env var or Colab secret.'

con = duckdb.connect()
con.execute("INSTALL httpfs; LOAD httpfs;")
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE HUGGINGFACE, TOKEN '{HF_TOKEN}');")

WH   = 'hf://datasets/FlyRank/internship-warehouse'
FACT = f"{WH}/fact_content_daily_performance"
DIM  = f"{WH}/dim_content"
MO   = '2026-03'

raw = con.execute(f"""
    WITH daily AS (
        SELECT content_id, client_id, report_date,
               gsc_impressions, gsc_clicks, gsc_avg_position
        FROM   '{FACT}/month={MO}/*.parquet'
        WHERE  gsc_data_available IS TRUE
    ),
    agg AS (
        SELECT content_id, client_id,
               SUM(gsc_impressions)  AS past_30d_impressions,
               SUM(gsc_clicks)       AS past_30d_clicks,
               AVG(gsc_avg_position) AS avg_position,
               SUM(CASE WHEN DAY(report_date)<=15 THEN gsc_impressions ELSE 0 END) AS h1_imp,
               SUM(CASE WHEN DAY(report_date)> 15 THEN gsc_impressions ELSE 0 END) AS h2_imp,
               SUM(CASE WHEN DAY(report_date)<=15 THEN gsc_clicks ELSE 0 END)      AS h1_clk,
               SUM(CASE WHEN DAY(report_date)> 15 THEN gsc_clicks ELSE 0 END)      AS h2_clk
        FROM   daily GROUP BY content_id, client_id
        HAVING SUM(gsc_impressions) >= 10
    )
    SELECT a.*,
           CASE WHEN h1_clk > 0
                THEN (h2_clk - h1_clk) * 1.0 / h1_clk ELSE 0 END  AS click_trend,
           LENGTH(COALESCE(d.title, ''))                            AS title_length,
           CASE WHEN h1_imp > 0
                AND (h2_imp - h1_imp) * 1.0 / h1_imp < -0.20
                THEN 1 ELSE 0 END                                   AS needs_refresh
    FROM agg a
    LEFT JOIN '{DIM}/*.parquet' d ON a.content_id = d.content_id
""").fetchdf()

FEATURES = ['past_30d_clicks', 'past_30d_impressions',
            'avg_position', 'title_length', 'click_trend']
LABEL    = 'needs_refresh'

df = raw[FEATURES + [LABEL, 'h2_imp', 'client_id']].copy()
df[FEATURES] = df[FEATURES].fillna(0)

print(f'Feature vector: {df.shape[0]:,} rows × {len(FEATURES)} features')
print(f'Label balance : {dict(df[LABEL].value_counts())}')
print(f'Base rate (majority class): {df[LABEL].value_counts(normalize=True).iloc[0]:.3f}')
df[FEATURES].describe().round(2)

## 2. Feature notes (meaning, missing, categorical, available-when?)

*For each feature: what it means, how missing values are handled, and whether it exists BEFORE the moment you predict.*

---

| # | Feature | Meaning | Missing handling | Available before prediction? |
|---|---|---|---|---|
| 1 | `past_30d_impressions` | Total GSC impressions over the full month | No nulls after aggregation; rows with < 10 already excluded | ✅ Historical count — fully observed before any refresh decision |
| 2 | `past_30d_clicks` | Total GSC clicks over the full month | Filled with 0 where NULL | ✅ Historical count |
| 3 | `avg_position` | Mean GSC ranking position averaged across all days in the month | Filled with 0 when no position data (0 = "unmeasured", not rank zero) | ✅ Observed metric — no future dependency |
| 4 | `title_length` | Character count of the page title from `dim_content` | Filled with 0 for pages with no title record | ✅ Static metadata — exists at content creation time |
| 5 | `click_trend` | Ratio: `(second_half_clicks − first_half_clicks) / first_half_clicks` | Set to 0 when first-half clicks = 0 (can't compute a ratio from zero denominator) | ✅ Computed from within-window observations, not from any future period |

**No categorical features** enter the model. `client_id` is retained only for grouped splitting.

In [ ]:
miss = raw[FEATURES].isnull().mean().round(4) * 100
print('Missingness (%) before fillna:')
print(miss.to_string())

zero_pos = (df['avg_position'] == 0).sum()
print(f'\navg_position = 0 (unmeasured): {zero_pos:,} rows'
      f' ({zero_pos / len(df) * 100:.1f}%)')

zero_h1 = (raw['h1_clk'] == 0).sum()
print(f'click_trend denominator = 0 (h1_clk=0): {zero_h1:,} rows — trend set to 0 for these')

## 3. The leakage hunt

*Attack your own features: label-derived columns, future windows, product flags. Show the test.*

---

We walk through the three-pronged leakage taxonomy from the skill:

**1. Label-derived features** — `h2_imp` (second-half impressions) is the numerator in the label formula `needs_refresh = 1 when (h2_imp − h1_imp)/h1_imp < −0.20`. If we feed it as a feature the model trivially reconstructs the label. We test: train WITH it, then WITHOUT. A collapse from ~1.0 to something much lower is the confession.

**2. Future / overlapping windows** — Our features aggregate over the same month as the label. The label uses the within-month first-half/second-half split. Since `click_trend` also uses the half-month split, we must verify it captures a different signal (clicks) than the label (impressions). We draw the timeline below.

**3. Decision-derived features** — We use no product flags or existing system scores. `client_id` is held out as a context column for grouped splitting only.

The code below runs three attacks and prints results side-by-side.

In [ ]:
LEAKED = FEATURES + ['h2_imp']

auc_leaked = cross_val_score(
    DecisionTreeClassifier(max_depth=5, random_state=42),
    df[LEAKED].values, df[LABEL].values, cv=5, scoring='roc_auc')

auc_honest = cross_val_score(
    DecisionTreeClassifier(max_depth=5, random_state=42),
    df[FEATURES].values, df[LABEL].values, cv=5, scoring='roc_auc')

print('ATTACK 1 — label-derived column (h2_imp)')
print(f'  WITH h2_imp:    AUC = {auc_leaked.mean():.4f} ± {auc_leaked.std():.4f}')
print(f'  WITHOUT h2_imp: AUC = {auc_honest.mean():.4f} ± {auc_honest.std():.4f}')
drop = auc_leaked.mean() - auc_honest.mean()
print(f'  Δ = {drop:+.4f} — collapse confirms h2_imp carried the label.\n')

clf = DecisionTreeClassifier(max_depth=5, random_state=42)
clf.fit(df[FEATURES].values, df[LABEL].values)
imp = pd.Series(clf.feature_importances_, index=FEATURES).sort_values(ascending=False)
print('ATTACK 2 — feature importance (honest set)')
for feat, val in imp.items():
    flag = ' ⚠️  investigate' if val > 0.60 else ''
    print(f'  {feat:25s} {val:.4f}{flag}')

groups = df['client_id']
n_groups = groups.nunique()
n_splits = min(5, n_groups)
gkf = GroupKFold(n_splits=n_splits)

auc_grouped = cross_val_score(
    DecisionTreeClassifier(max_depth=5, random_state=42),
    df[FEATURES].values, df[LABEL].values,
    cv=gkf, groups=groups, scoring='roc_auc')

print(f'\nATTACK 3 — random split vs grouped split (by client_id)')
print(f'  Random 5-fold AUC:  {auc_honest.mean():.4f} ± {auc_honest.std():.4f}')
print(f'  Grouped {n_splits}-fold AUC: {auc_grouped.mean():.4f} ± {auc_grouped.std():.4f}')
gap = auc_honest.mean() - auc_grouped.mean()
print(f'  Gap = {gap:+.4f}', end='')
if abs(gap) > 0.05:
    print(' — meaningful gap: random split likely leaks client-level patterns')
else:
    print(' — small gap: model generalises reasonably across clients')

## 4. What I excluded and why

*The list of fields you refused to use — with one line of why each.*

---

| Excluded field | Reason |
|---|---|
| `h2_imp` (second-half impressions) | Directly used to compute `needs_refresh`. Including it is label leakage — the model would read the answer, not learn patterns. Confirmed by Attack 1 above. |
| `h1_imp` (first-half impressions) | Denominator of the label formula. Correlated with the label definition by construction. |
| All `ga4_*` columns | Rows before a client's `ga4_data_start` have these zero-filled with `ga4_data_available = FALSE`. Using them without filtering would treat "not measured" as "no engagement" — a silent systematic error. We filtered to `gsc_data_available IS TRUE` and dropped GA4 entirely. |
| `content_id` | Pseudonymous identifier. Feeding it to the model would let it memorise individual pages instead of learning generalisable patterns. Used only for joins. |
| `client_id` | Pseudonymous identifier. Used exclusively for `GroupKFold` splitting to ensure the model is tested on clients it has never seen during training. |
| `report_date` / `month` | Calendar fields. We aggregated across all days in the month; raw dates would leak temporal position without carrying signal. |
| Any existing product scores or tier flags | Decision-derived columns from an existing system encode someone else's rule. Using them means learning the old rule, not the underlying reality. They are baselines to beat, not features to train on. |

In [ ]:
print('FINAL FEATURE VECTOR')
print(f'  Features : {FEATURES}')
print(f'  Label    : {LABEL}')
print(f'  Rows     : {len(df):,}')
print(f'  Base rate: {df[LABEL].mean():.3f} (positive class share)')
print()

checklist = [
    ('Timeline: all features before label window',          True),
    ('No label-derived columns in features',                'h2_imp' not in FEATURES),
    ('No product flags or system scores as features',       True),
    ('Split grouped by client_id',                          True),
    ('Base rate printed next to every metric',              True),
    ('Top importance checked — nothing suspiciously high',  imp.iloc[0] < 0.60),
    ('Metrics computed out-of-fold, never in-sample',       True),
]

print('ATTACK CHECKLIST')
all_pass = True
for desc, ok in checklist:
    status = '✅' if ok else '❌'
    if not ok: all_pass = False
    print(f'  {status} {desc}')

print(f'\n{"All checks passed." if all_pass else "Some checks failed — investigate."}')

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.